# Financial Agent EDA

Notebook de análise exploratória para perceber rapidamente o universo de ativos, a qualidade dos dados, os perfis de risco/retorno e as relações entre os ativos.

Objetivos:
- confirmar a cobertura temporal dos datasets
- medir performance e risco por ativo
- identificar correlações e concentração de exposição
- preparar insights para o MVP e para o pitch

In [15]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.float_format = '{:,.2f}'.format

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DAILY_DIR = BASE_DIR / 'data' / 'raw' / 'daily'
HOURLY_DIR = BASE_DIR / 'data' / 'raw' / 'hourly'

assert DAILY_DIR.exists(), f'Missing directory: {DAILY_DIR}'
assert HOURLY_DIR.exists(), f'Missing directory: {HOURLY_DIR}'

DAILY_DIR, HOURLY_DIR

(PosixPath('/Users/pedrofs/EY_Challenge/Financial-Agent/data/raw/daily'),
 PosixPath('/Users/pedrofs/EY_Challenge/Financial-Agent/data/raw/hourly'))

In [16]:
def load_market_data(folder: Path, time_col: str) -> dict[str, pd.DataFrame]:
    frames = {}
    for csv_path in sorted(folder.glob('*.csv')):
        frame = pd.read_csv(csv_path)
        frame[time_col] = pd.to_datetime(frame[time_col])
        frame = frame.sort_values(time_col).reset_index(drop=True)
        frames[csv_path.stem] = frame
    return frames


def annualized_return(returns: pd.Series, periods_per_year: int = 252) -> float:
    clean = returns.dropna()
    if clean.empty:
        return np.nan
    compounded = (1 + clean).prod()
    return compounded ** (periods_per_year / len(clean)) - 1


def annualized_volatility(returns: pd.Series, periods_per_year: int = 252) -> float:
    clean = returns.dropna()
    if clean.empty:
        return np.nan
    return clean.std() * np.sqrt(periods_per_year)


def sharpe_ratio(returns: pd.Series, risk_free_rate: float = 0.0, periods_per_year: int = 252) -> float:
    clean = returns.dropna()
    if clean.empty:
        return np.nan
    excess = clean - risk_free_rate / periods_per_year
    vol = annualized_volatility(excess, periods_per_year)
    if not vol or np.isnan(vol):
        return np.nan
    return annualized_return(excess, periods_per_year) / vol


def max_drawdown(price_series: pd.Series) -> float:
    running_max = price_series.cummax()
    drawdown = price_series / running_max - 1
    return drawdown.min()


daily_frames = load_market_data(DAILY_DIR, 'Date')
hourly_frames = load_market_data(HOURLY_DIR, 'Datetime')

list(daily_frames)[:5], list(hourly_frames)[:5]

(['AAPL', 'AMZN', 'BTC-USD', 'CDR.WA', 'EH'],
 ['AAPL', 'AMZN', 'BTC-USD', 'CDR.WA', 'EH'])

## 1. Cobertura e qualidade dos dados

In [17]:
coverage_rows = []

for ticker, frame in daily_frames.items():
    coverage_rows.append({
        'Ticker': ticker,
        'Dataset': 'Daily',
        'Start': frame['Date'].min(),
        'End': frame['Date'].max(),
        'Rows': len(frame),
        'Missing Values': int(frame.isna().sum().sum()),
    })

for ticker, frame in hourly_frames.items():
    coverage_rows.append({
        'Ticker': ticker,
        'Dataset': 'Hourly',
        'Start': frame['Datetime'].min(),
        'End': frame['Datetime'].max(),
        'Rows': len(frame),
        'Missing Values': int(frame.isna().sum().sum()),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values(['Dataset', 'Ticker']).reset_index(drop=True)
coverage_df['Start Plot'] = pd.to_datetime(coverage_df['Start'], utc=True).dt.tz_localize(None)
coverage_df['End Plot'] = pd.to_datetime(coverage_df['End'], utc=True).dt.tz_localize(None)
coverage_df


,Ticker,Dataset,Start,End,Rows,Missing Values,Start Plot,End Plot
0,AAPL,Daily,2021-05-20 00:00:00,2026-05-19 00:00:00,1255,0,2021-05-20 00:00:00,2026-05-19 00:00:00
1,AMZN,Daily,2021-05-20 00:00:00,2026-05-19 00:00:00,1255,0,2021-05-20 00:00:00,2026-05-19 00:00:00
2,BTC-USD,Daily,2021-05-20 00:00:00,2026-05-20 00:00:00,1827,0,2021-05-20 00:00:00,2026-05-20 00:00:00
3,CDR.WA,Daily,2021-05-20 00:00:00,2026-05-20 00:00:00,1251,0,2021-05-20 00:00:00,2026-05-20 00:00:00
4,EH,Daily,2021-05-20 00:00:00,2026-05-19 00:00:00,1255,0,2021-05-20 00:00:00,2026-05-19 00:00:00
5,ETH-USD,Daily,2021-05-20 00:00:00,2026-05-20 00:00:00,1827,0,2021-05-20 00:00:00,2026-05-20 00:00:00
6,GOOGL,Daily,2021-05-20 00:00:00,2026-05-19 00:00:00,1255,0,2021-05-20 00:00:00,2026-05-19 00:00:00
7,MSFT,Daily,2021-05-20 00:00:00,2026-05-19 00:00:00,1255,0,2021-05-20 00:00:00,2026-05-19 00:00:00
8,NXE,Daily,2021-05-20 00:00:00,2026-05-19 00:00:00,1255,0,2021-05-20 00:00:00,2026-05-19 00:00:00
9,SPY,Daily,2021-05-20 00:00:00,2026-05-19 00:00:00,1255,0,2021-05-20 00:00:00,2026-05-19 00:00:00


In [18]:
coverage_chart = px.timeline(
    coverage_df,
    x_start='Start Plot',
    x_end='End Plot',
    y='Ticker',
    color='Dataset',
    title='Coverage timeline by asset and dataset',
)
coverage_chart.update_yaxes(autorange='reversed')
coverage_chart.update_layout(height=600, template='plotly_white')
coverage_chart.show()


## 2. Preços normalizados e evolução histórica

In [19]:
daily_prices = []
for ticker, frame in daily_frames.items():
    price_slice = frame[['Date', 'Adj Close']].rename(columns={'Adj Close': ticker}).set_index('Date')
    daily_prices.append(price_slice)

price_df = pd.concat(daily_prices, axis=1).sort_index()
normalized_price_df = price_df.divide(price_df.iloc[0]).mul(100)

fig = px.line(
    normalized_price_df.reset_index(),
    x='Date',
    y=normalized_price_df.columns,
    title='Normalized performance (base 100)',
)
fig.update_layout(height=650, template='plotly_white', legend_title='Ticker')
fig.show()

## 3. Tabela de performance e risco

In [20]:
summary_rows = []

for ticker, frame in daily_frames.items():
    returns = frame['Adj Close'].pct_change()
    summary_rows.append({
        'Ticker': ticker,
        'Start': frame['Date'].min().date(),
        'End': frame['Date'].max().date(),
        'Start Price': frame['Adj Close'].iloc[0],
        'End Price': frame['Adj Close'].iloc[-1],
        'Total Return %': (frame['Adj Close'].iloc[-1] / frame['Adj Close'].iloc[0] - 1) * 100,
        'Annualized Return %': annualized_return(returns) * 100,
        'Annualized Volatility %': annualized_volatility(returns) * 100,
        'Sharpe': sharpe_ratio(returns),
        'Max Drawdown %': max_drawdown(frame['Adj Close']) * 100,
        'Best Day %': returns.max() * 100,
        'Worst Day %': returns.min() * 100,
        'Average Volume': frame['Volume'].mean(),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Total Return %', ascending=False).reset_index(drop=True)
summary_df

,Ticker,Start,End,Start Price,End Price,Total Return %,Annualized Return %,Annualized Volatility %,Sharpe,Max Drawdown %,Best Day %,Worst Day %,Average Volume
0,GOOGL,2021-05-20,2026-05-19,114.40,387.66,238.86,27.79,31.25,0.89,-44.32,10.22,-9.51,"32,088,664.38"
1,AAPL,2021-05-20,2026-05-19,124.10,298.97,140.92,19.33,27.44,0.70,-33.36,15.33,-9.25,"65,563,427.25"
2,NXE,2021-05-20,2026-05-19,4.56,10.53,130.92,18.32,57.82,0.32,-54.28,14.75,-15.88,"5,040,249.96"
3,BTC-USD,2021-05-20,2026-05-20,"40,782.74","77,508.46",90.05,9.27,45.30,0.20,-76.63,14.54,-15.97,"35,205,179,496.39"
4,SPY,2021-05-20,2026-05-19,387.86,733.73,89.17,13.67,17.06,0.80,-24.50,10.50,-5.85,"75,718,968.76"
5,MSFT,2021-05-20,2026-05-19,237.06,417.42,76.08,12.04,26.38,0.46,-37.15,10.13,-9.99,"26,007,540.64"
6,AMZN,2021-05-20,2026-05-19,162.38,259.34,59.71,9.87,35.43,0.28,-56.15,13.54,-14.05,"55,693,577.29"
7,CDR.WA,2021-05-20,2026-05-20,165.72,257.30,55.26,9.27,41.52,0.22,-62.32,13.47,-10.91,"390,473.89"
8,ETH-USD,2021-05-20,2026-05-20,"2,784.29","2,129.77",-23.51,-3.63,60.82,-0.06,-79.35,25.31,-17.46,"18,480,434,142.67"
9,EH,2021-05-20,2026-05-19,22.81,9.41,-58.75,-16.30,84.79,-0.19,-91.81,37.45,-19.48,"1,155,201.59"


In [21]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Total return %', 'Annualized volatility %'))

fig.add_trace(
    go.Bar(x=summary_df['Ticker'], y=summary_df['Total Return %'], marker_color='#2ca02c', name='Total Return %'),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(x=summary_df['Ticker'], y=summary_df['Annualized Volatility %'], marker_color='#d62728', name='Annualized Volatility %'),
    row=1,
    col=2,
)

fig.update_layout(height=500, template='plotly_white', title='Performance vs risk snapshot', showlegend=False)
fig.show()

## 4. Correlação entre ativos

In [22]:
returns_df = price_df.pct_change(fill_method=None)
corr_df = returns_df.corr()

heatmap = px.imshow(
    corr_df,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Daily returns correlation matrix',
)
heatmap.update_layout(height=750, template='plotly_white')
heatmap.show()

corr_df['SPY'].sort_values(ascending=False).to_frame('Correlation with SPY')

,Correlation with SPY
SPY,1.00
AAPL,0.77
MSFT,0.72
AMZN,0.72
GOOGL,0.68
NXE,0.48
ETH-USD,0.45
BTC-USD,0.43
UDMY,0.40
EH,0.35


## 5. Mapa risco-retorno

In [23]:
risk_return_df = summary_df[['Ticker', 'Annualized Return %', 'Annualized Volatility %', 'Sharpe', 'Max Drawdown %']].copy()
risk_return_df['Sharpe Size'] = risk_return_df['Sharpe'].fillna(0).clip(lower=0) + 0.05

scatter = px.scatter(
    risk_return_df,
    x='Annualized Volatility %',
    y='Annualized Return %',
    size='Sharpe Size',
    color='Max Drawdown %',
    text='Ticker',
    hover_data=['Sharpe'],
    title='Risk-return map',
    color_continuous_scale='RdYlGn_r',
)
scatter.update_traces(textposition='top center')
scatter.update_layout(height=650, template='plotly_white')
scatter.show()


## 6. Drawdowns dos principais ativos

In [24]:
selected_assets = ['SPY', 'AAPL', 'MSFT', 'GOOGL', 'BTC-USD', 'ETH-USD']
drawdown_df = pd.DataFrame(index=price_df.index)

for ticker in selected_assets:
    if ticker in price_df.columns:
        running_max = price_df[ticker].cummax()
        drawdown_df[ticker] = price_df[ticker] / running_max - 1

fig = px.line(
    drawdown_df.reset_index(),
    x='Date',
    y=drawdown_df.columns,
    title='Drawdown comparison',
)
fig.update_layout(height=650, template='plotly_white', yaxis_tickformat='.0%')
fig.show()

## 7. Sinais recentes com dados horários

In [25]:
hourly_rows = []

for ticker, frame in hourly_frames.items():
    prices = frame['Adj Close']
    latest = prices.iloc[-1]
    ret_24h = (latest / prices.iloc[-24] - 1) * 100 if len(prices) >= 24 else np.nan
    ret_5d = (latest / prices.iloc[-35] - 1) * 100 if len(prices) >= 35 else np.nan
    ret_total = (latest / prices.iloc[0] - 1) * 100
    hourly_rows.append({
        'Ticker': ticker,
        'Start': frame['Datetime'].min(),
        'End': frame['Datetime'].max(),
        'Rows': len(frame),
        'Latest Price': latest,
        '24h Return %': ret_24h,
        '5D Return %': ret_5d,
        'Period Return %': ret_total,
    })

hourly_summary_df = pd.DataFrame(hourly_rows).sort_values('24h Return %', ascending=False).reset_index(drop=True)
hourly_summary_df

,Ticker,Start,End,Rows,Latest Price,24h Return %,5D Return %,Period Return %
0,MSFT,2026-02-24 14:30:00+00:00,2026-05-19 19:30:00+00:00,420,417.56,2.05,3.60,7.56
1,BTC-USD,2026-03-22 00:00:00+00:00,2026-05-20 10:00:00+00:00,1414,"77,497.26",1.12,0.54,12.46
2,ETH-USD,2026-03-22 00:00:00+00:00,2026-05-20 10:00:00+00:00,1414,"2,129.77",1.01,-0.21,2.08
3,AAPL,2026-02-24 14:30:00+00:00,2026-05-19 19:30:00+00:00,420,298.98,0.20,1.30,9.25
4,CDR.WA,2026-02-23 08:00:00+00:00,2026-05-20 10:00:00+00:00,533,257.30,-1.04,-1.76,6.19
5,UDMY,2026-02-24 14:30:00+00:00,2026-05-08 19:30:00+00:00,371,4.62,-1.81,-4.55,-2.74
6,SPY,2026-02-24 14:30:00+00:00,2026-05-19 19:30:00+00:00,420,733.79,-1.84,-0.61,6.91
7,GOOGL,2026-02-24 14:30:00+00:00,2026-05-19 19:30:00+00:00,420,387.70,-3.05,-1.82,25.31
8,AMZN,2026-02-24 14:30:00+00:00,2026-05-19 19:30:00+00:00,420,259.42,-3.13,-2.28,24.87
9,EH,2026-02-24 14:30:00+00:00,2026-05-19 19:30:00+00:00,420,9.41,-4.61,-5.71,-22.57


In [26]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('24h return %', '5D return %'))

fig.add_trace(
    go.Bar(x=hourly_summary_df['Ticker'], y=hourly_summary_df['24h Return %'], marker_color='#1f77b4'),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(x=hourly_summary_df['Ticker'], y=hourly_summary_df['5D Return %'], marker_color='#ff7f0e'),
    row=1,
    col=2,
)

fig.update_layout(height=500, template='plotly_white', title='Recent short-term momentum', showlegend=False)
fig.show()